In [1]:
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
import seaborn as sns
import os
import glob
from datetime import datetime
from datetime import timedelta
from matplotlib import pyplot as plt
import matplotlib.dates as md
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings
from matplotlib import cm
import matplotlib.dates as mdates
from scipy.interpolate import interp2d
warnings.filterwarnings('ignore')
#import datetime
import scipy.ndimage as ndimage
from matplotlib import cm
import geopy.distance
#import matplotlib as mpl
from scipy.interpolate import interp1d
from sklearn.linear_model import LinearRegression
from shapely.geometry import Point
import geopandas as gpd
from geopandas import GeoDataFrame
import leafmap
import plotly.express as px
import matplotlib as mpl
import xarray as xr
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D
import math
#import pysplit
import netCDF4
import xarray as xr
import matplotlib.ticker as mticker

Cannot find header.dxf (GDAL_DATA is not defined)


In [2]:
csv_file_path = 'C:/Users/taiwoajayi/\OneDrive - University of Arizona/Arizona_ozone/Zip/Combined_state_Data.csv'

# Read the CSV file into a pandas DataFrame
MDA_O3 = pd.read_csv(csv_file_path, sep = ',', skiprows=0)
MDA_O3['Timestamp'] = pd.to_datetime(MDA_O3['Date Local'])
MD_O3 = MDA_O3.rename(columns={'1st Max Value': 'Sample Measurement', 'Arithmetic Mean': 'MDL'})
MD_O3['Sample Measurement'] = MD_O3['Sample Measurement'] * 1000

#MDA_O3['Sample Measurement'] = MDA_O3.apply(lambda row: row['Sample Measurement']*0.5*row['MDL'] if row['Sample Measurement'] < row['MDL'] else row['Sample Measurement'], axis=1)
MD_O3
# Define the columns of interest
columns_of_interest = ['Date Local', 'State Code', 'County Code', 'Site Num', 'Latitude', 'Longitude', 'Timestamp', 'Sample Measurement', 'MDL']

In [3]:
# Assuming location_df_daily is your DataFrame
condition = ~((MD_O3['Timestamp'] >= pd.to_datetime('2020-03-15')) & (MD_O3['Timestamp'] <= pd.to_datetime('2020-08-31')))
filtered_df_MDA8 = MD_O3[condition]
filtered_df_MDA8

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Sample Duration,...,Method Code,Method Name,Local Site Name,Address,State Name,County Name,City Name,CBSA Name,Date of Last Change,Timestamp
0,4,3,8001,44201,1,32.009410,-109.389060,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,47.0,INSTRUMENTAL - ULTRA VIOLET,Chiricahua NM - Entrance Station,CHIRICAHUA NATIONAL MOUMENT,Arizona,Cochise,Not in a city,"Sierra Vista-Douglas, AZ",2023-02-05,1997-01-01
1,4,3,8001,44201,1,32.009410,-109.389060,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,47.0,INSTRUMENTAL - ULTRA VIOLET,Chiricahua NM - Entrance Station,CHIRICAHUA NATIONAL MOUMENT,Arizona,Cochise,Not in a city,"Sierra Vista-Douglas, AZ",2023-02-05,1997-01-02
2,4,3,8001,44201,1,32.009410,-109.389060,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,47.0,INSTRUMENTAL - ULTRA VIOLET,Chiricahua NM - Entrance Station,CHIRICAHUA NATIONAL MOUMENT,Arizona,Cochise,Not in a city,"Sierra Vista-Douglas, AZ",2023-02-05,1997-01-03
3,4,3,8001,44201,1,32.009410,-109.389060,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,47.0,INSTRUMENTAL - ULTRA VIOLET,Chiricahua NM - Entrance Station,CHIRICAHUA NATIONAL MOUMENT,Arizona,Cochise,Not in a city,"Sierra Vista-Douglas, AZ",2023-02-05,1997-01-04
4,4,3,8001,44201,1,32.009410,-109.389060,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,47.0,INSTRUMENTAL - ULTRA VIOLET,Chiricahua NM - Entrance Station,CHIRICAHUA NATIONAL MOUMENT,Arizona,Cochise,Not in a city,"Sierra Vista-Douglas, AZ",2023-02-05,1997-01-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2767789,49,57,1003,44201,1,41.303614,-111.987871,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,87.0,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,Harrisville,"425 W 2550 NORTH, OGDEN, UTAH",Utah,Weber,Harrisville,"Ogden-Clearfield, UT",2024-05-26,2022-12-27
2767790,49,57,1003,44201,1,41.303614,-111.987871,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,87.0,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,Harrisville,"425 W 2550 NORTH, OGDEN, UTAH",Utah,Weber,Harrisville,"Ogden-Clearfield, UT",2024-05-26,2022-12-28
2767791,49,57,1003,44201,1,41.303614,-111.987871,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,87.0,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,Harrisville,"425 W 2550 NORTH, OGDEN, UTAH",Utah,Weber,Harrisville,"Ogden-Clearfield, UT",2024-05-26,2022-12-29
2767792,49,57,1003,44201,1,41.303614,-111.987871,WGS84,Ozone,8-HR RUN AVG BEGIN HOUR,...,87.0,INSTRUMENTAL - ULTRA VIOLET ABSORPTION,Harrisville,"425 W 2550 NORTH, OGDEN, UTAH",Utah,Weber,Harrisville,"Ogden-Clearfield, UT",2024-05-26,2022-12-30


In [4]:
MDA_2 = filtered_df_MDA8[(filtered_df_MDA8['Timestamp'].dt.year >= 2001) & (filtered_df_MDA8['Timestamp'].dt.year <= 2022)]

In [5]:
MDA_20 = filtered_df_MDA8[(filtered_df_MDA8['Timestamp'].dt.year >= 2010) & (filtered_df_MDA8['Timestamp'].dt.year <= 2022)]

In [ ]:
# Define zones with corresponding locations
zones = {
    'Northwest': ['Tangerine', 'Coach Line'],
    'Urban Core': ['Craycroft', 'Childrens Park', 'Rose Elementary'],
    'South/Southeast': ['Saguaro Park', 'Fairgrounds', 'Green Valley']
}

# Define longitude and latitude ranges for each location
locations = {
    'Childrens Park': (-110.9823, 32.29515),
    'Green Valley': (-110.99644, 31.87952),
    'Coach Line': (-111.12716, 32.38082),
    'Rose Elementary': (-110.980134, 32.172995),
    'Fairgrounds': (-110.774357, 32.04767),
    'Tangerine': (-111.06352, 32.425261),
    'Craycroft': (-110.878067, 32.204411),
    'Saguaro Park': (-110.737116, 32.174538)
}

# Extract data and group by zones
def extract_data_by_zone(data, locations, zones):
    zone_data = {zone: [] for zone in zones}  # Initialize empty lists for each zone

    for site, (lon, lat) in locations.items():
        # Filter data for the site
        data_site = data[(data['Longitude'] == lon) & (data['Latitude'] == lat)].copy()
        
        # Ensure Timestamp is in datetime format
        data_site['Timestamp'] = pd.to_datetime(data_site['Timestamp'])
        
        # Assign the data to the appropriate zone
        for zone, site_list in zones.items():
            if site in site_list:
                zone_data[zone].append(data_site)

    # Combine all data within each zone
    for zone in zone_data:
        if zone_data[zone]:  # Ensure there's data
            zone_data[zone] = pd.concat(zone_data[zone])

    return zone_data

# Compute yearly mean or median for each zone
def compute_zone_aggregates(zone_data, method='mean'):
    zone_aggregates = {}

    for zone, df in zone_data.items():
        df['Year'] = df['Timestamp'].dt.year  # Extract year

        if method == 'mean':
            yearly_agg = df.groupby('Year')['Sample Measurement'].mean().reset_index()
        elif method == 'median':
            yearly_agg = df.groupby('Year')['Sample Measurement'].median().reset_index()
        else:
            raise ValueError("Invalid method. Use 'mean' or 'median'.")

        zone_aggregates[zone] = yearly_agg

    return zone_aggregates

# Plot function for zones
def plot_zonal_time_series(zone_aggregates):
    plt.figure(figsize=(15, 6))

    for zone, df in zone_aggregates.items():
        plt.plot(df['Year'], df['Sample Measurement'], marker='o', linestyle='-', label=zone)

    # Formatting
    plt.xlabel('Year', fontsize=16)
    plt.ylabel('Mean MDA8 O$_3$ (ppb)', fontsize=16)
    plt.tick_params(axis='both', which='major', labelsize=16)
    #plt.title('Yearly Mean Sample Measurement (2001-2020) by Zone')
    plt.legend(fontsize=16)
    plt.grid(True)

    # Format x-axis to show only integer years
    ax = plt.gca()
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.show()

# Main Execution
zone_data = extract_data_by_zone(MD_O3, locations, zones)
zone_aggregates = compute_zone_aggregates(zone_data, method='mean')  # Change to 'median' if needed
plot_zonal_time_series(zone_aggregates)


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau, linregress

# Define zones with corresponding locations
zones = {
    'Upwind': ['Tangerine', 'Coach Line'],
    'Urban Core': ['Craycroft', 'Childrens Park', 'Rose Elementary'],
    'Downwind': ['Saguaro Park', 'Fairgrounds', 'Green Valley']
}

# Define longitude and latitude ranges for each location
locations = {
    'Childrens Park': (-110.9823, 32.29515),
    'Green Valley': (-110.99644, 31.87952),
    'Coach Line': (-111.12716, 32.38082),
    'Rose Elementary': (-110.980134, 32.172995),
    'Fairgrounds': (-110.774357, 32.04767),
    'Tangerine': (-111.06352, 32.425261),
    'Craycroft': (-110.878067, 32.204411),
    'Saguaro Park': (-110.737116, 32.174538)
}

# Extract data and group by zones
def extract_data_by_zone(data, locations, zones):
    zone_data = {zone: [] for zone in zones}  # Initialize empty lists for each zone

    for site, (lon, lat) in locations.items():
        # Filter data for the site
        data_site = data[(data['Longitude'] == lon) & (data['Latitude'] == lat)].copy()
        
        # Ensure Timestamp is in datetime format
        data_site['Timestamp'] = pd.to_datetime(data_site['Timestamp'])
        
        # Assign the data to the appropriate zone
        for zone, site_list in zones.items():
            if site in site_list:
                zone_data[zone].append(data_site)

    # Combine all data within each zone
    for zone in zone_data:
        if zone_data[zone]:  # Ensure there's data
            zone_data[zone] = pd.concat(zone_data[zone])

    return zone_data

# Compute monthly mean for each zone
def compute_monthly_zone_aggregates(zone_data):
    zone_aggregates = {}

    for zone, df in zone_data.items():
        df['Year'] = df['Timestamp'].dt.year
        df['Month'] = df['Timestamp'].dt.month
        
        # Compute monthly mean
        monthly_agg = df.groupby(['Year', 'Month'])['Sample Measurement'].mean().reset_index()
        
        zone_aggregates[zone] = monthly_agg

    return zone_aggregates

# Perform Mann-Kendall and Linear Regression Trend Tests
def trend_tests(zone_aggregates):
    trend_results = {}

    for zone, df in zone_aggregates.items():
        df['Time'] = df['Year'] + (df['Month'] - 1) / 12.0  # Convert Year-Month to fractional year
        
        # Mann-Kendall Test
        tau, mk_p_value = kendalltau(df['Time'], df['Sample Measurement'])

        # Linear Regression
        slope, intercept, r_value, p_value, std_err = linregress(df['Time'], df['Sample Measurement'])

        trend_results[zone] = {
            'Mann-Kendall Tau': tau,
            'Mann-Kendall p-value': mk_p_value,
            'Linear Regression Slope': slope,
            'Linear Regression p-value': p_value,
            'R-squared': r_value**2
        }

    return trend_results

# Main Execution
zone_data = extract_data_by_zone(MD_O3, locations, zones)
zone_aggregates = compute_monthly_zone_aggregates(zone_data)  # Using monthly data
trend_results = trend_tests(zone_aggregates)

# Convert trend results to a DataFrame for better readability
trend_results_df = pd.DataFrame.from_dict(trend_results, orient='index')
trend_results_df


In [ ]:
from scipy.stats import kendalltau, linregress

# Define zones with corresponding locations
zones = {
    'Upwind': ['Tangerine', 'Coach Line'],
    'Urban Core': ['Craycroft', 'Childrens Park', 'Rose Elementary'],
    'Downwind': ['Saguaro Park', 'Fairgrounds', 'Green Valley']
}

# Define longitude and latitude ranges for each location
locations = {
    'Childrens Park': (-110.9823, 32.29515),
    'Green Valley': (-110.99644, 31.87952),
    'Coach Line': (-111.12716, 32.38082),
    'Rose Elementary': (-110.980134, 32.172995),
    'Fairgrounds': (-110.774357, 32.04767),
    'Tangerine': (-111.06352, 32.425261),
    'Craycroft': (-110.878067, 32.204411),
    'Saguaro Park': (-110.737116, 32.174538)
}

# Define time periods
time_periods = {
    "2001-2005": (2001, 2005),
    "2006-2010": (2006, 2010),
    "2011-2015": (2011, 2015),
    "2016-2020": (2016, 2019),
    "2020-2022": (2020, 2022)
}

# Define seasons
seasons = {
    "Winter": [12, 1, 2],  # December, January, February
    "Spring": [3, 4, 5],  # March, April, May
    "Dry Summer": [6],  # June
    "Monsoon Summer": [7, 8],  # July, August
    "Fall": [9, 10, 11]  # September, October, November
}

# Extract data and group by zones
def extract_data_by_zone(data, locations, zones):
    zone_data = {zone: [] for zone in zones}  # Initialize empty lists for each zone

    for site, (lon, lat) in locations.items():
        # Filter data for the site
        data_site = data[(data['Longitude'] == lon) & (data['Latitude'] == lat)].copy()
        
        # Ensure Timestamp is in datetime format
        data_site['Timestamp'] = pd.to_datetime(data_site['Timestamp'])
        
        # Assign the data to the appropriate zone
        for zone, site_list in zones.items():
            if site in site_list:
                zone_data[zone].append(data_site)

    # Combine all data within each zone
    for zone in zone_data:
        if zone_data[zone]:  # Ensure there's data
            zone_data[zone] = pd.concat(zone_data[zone])

    return zone_data

# Compute seasonal aggregates for each time period
def compute_seasonal_aggregates(zone_data):
    seasonal_aggregates = {}

    for zone, df in zone_data.items():
        df['Year'] = df['Timestamp'].dt.year
        df['Month'] = df['Timestamp'].dt.month

        for period, (start_year, end_year) in time_periods.items():
            df_period = df[(df['Year'] >= start_year) & (df['Year'] <= end_year)]

            for season, months in seasons.items():
                df_season = df_period[df_period['Month'].isin(months)]
                
                if not df_season.empty:
                    # Compute mean for each season
                    seasonal_mean = df_season.groupby(['Year', 'Month'])['Sample Measurement'].mean().reset_index()
                    seasonal_aggregates[(zone, period, season)] = seasonal_mean

    return seasonal_aggregates

# Perform Mann-Kendall and Linear Regression Trend Tests for each season and period
def seasonal_trend_tests(seasonal_aggregates):
    trend_results = []

    for (zone, period, season), df in seasonal_aggregates.items():
        df['Time'] = df['Year'] + (df['Month'] - 1) / 12.0  # Convert Year-Month to fractional year

        # Mann-Kendall Test
        tau, mk_p_value = kendalltau(df['Time'], df['Sample Measurement'])

        # Linear Regression
        slope, intercept, r_value, p_value, std_err = linregress(df['Time'], df['Sample Measurement'])

        # Determine trend classification
        if mk_p_value < 0.05 and p_value < 0.05:
            if tau > 0 and slope > 0:
                trend_description = "Increasing Trend"
            elif tau < 0 and slope < 0:
                trend_description = "Decreasing Trend"
            else:
                trend_description = "No Trend"
        else:
            trend_description = "No Trend"

        # Store results
        trend_results.append({
            'Zone': zone,
            'Time Period': period,
            'Season': season,
            'Mann-Kendall Tau': tau,
            'Mann-Kendall p-value': mk_p_value,
            'Linear Regression Slope': slope,
            'Linear Regression p-value': p_value,
            'R-squared': r_value**2,
            'Trend': trend_description
        })

    return pd.DataFrame(trend_results)

# Main Execution
zone_data = extract_data_by_zone(MD_O3, locations, zones)
seasonal_aggregates = compute_seasonal_aggregates(zone_data)  
seasonal_trend_results = seasonal_trend_tests(seasonal_aggregates)
seasonal_trend_results

In [ ]:
# Define zones with corresponding locations
zones = {
    'Upwind': ['Tangerine', 'Coach Line'],
    'Urban Core': ['Craycroft', 'Childrens Park', 'Rose Elementary'],
    'Downwind': ['Saguaro Park', 'Fairgrounds', 'Green Valley']
}

# Define longitude and latitude for each location
locations = {
    'Childrens Park': (-110.9823, 32.29515),
    'Green Valley': (-110.99644, 31.87952),
    'Coach Line': (-111.12716, 32.38082),
    'Rose Elementary': (-110.980134, 32.172995),
    'Fairgrounds': (-110.774357, 32.04767),
    'Tangerine': (-111.06352, 32.425261),
    'Craycroft': (-110.878067, 32.204411),
    'Saguaro Park': (-110.737116, 32.174538)
}

# Define year bins
year_bins = {
    '2001-2005': (2001, 2005),
    '2006-2010': (2006, 2010),
    '2011-2015': (2011, 2015),
    '2016-2020': (2016, 2019),
    '2020-2022': (2020, 2022)
}

# Define day order
day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

# Extract data for each location
def extract_data_by_zone(data, locations, zones):
    zone_data = {zone: [] for zone in zones}

    for site, (lon, lat) in locations.items():
        data_site = data[(data['Longitude'] == lon) & (data['Latitude'] == lat)].copy()
        data_site['Timestamp'] = pd.to_datetime(data_site['Timestamp'])

        for zone, site_list in zones.items():
            if site in site_list:
                zone_data[zone].append((site, data_site))

    return zone_data

# Compute daily means for each zone within each year bin
def compute_zone_binned_means(zone_data, year_bins):
    binned_zone_data = {bin_name: {} for bin_name in year_bins}

    for bin_name, (start_year, end_year) in year_bins.items():
        for zone, site_data in zone_data.items():
            daily_means = []

            for site, df in site_data:
                df['Year'] = df['Timestamp'].dt.year
                df['Day'] = df['Timestamp'].dt.day_name()
                df['Day'] = pd.Categorical(df['Day'], categories=day_order, ordered=True)

                # Filter by year bin
                df_filtered = df[(df['Year'] >= start_year) & (df['Year'] <= end_year)]

                # Compute daily mean
                daily_avg = df_filtered.groupby('Day')['Sample Measurement'].mean().reset_index()
                daily_means.append(daily_avg.set_index('Day'))  # Set index to merge easily

            # Compute overall mean across sites within the zone
            zone_mean = pd.concat(daily_means, axis=1).mean(axis=1, skipna=True).reset_index()
            zone_mean.columns = ['Day', 'Zone Mean']

            binned_zone_data[bin_name][zone] = zone_mean

    return binned_zone_data

# Plot daily means for each zone in binned years
def plot_binned_zone_means(binned_zone_data):
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(14, 10), sharex=False, sharey=False)
    axes = axes.flatten()
    axes[-1].set_visible(False)

    for ax, (bin_name, zones_data) in zip(axes, binned_zone_data.items()):
        for zone, df in zones_data.items():
            ax.plot(df['Day'], df['Zone Mean'], marker='o', linestyle='-', label=zone)

        ax.set_title(f'{bin_name}', fontsize=16)
        ax.set_ylabel('Mean MDA8 O$_3$ (ppb)', fontsize=16)
        ax.tick_params(axis='both', which='major', labelsize=16)
        ax.grid(True)
        ax.set_xticklabels(df['Day'], rotation=45)
        ax.set_xlabel('Day of the Week', fontsize=16)

    # Set common x-axis label

    # Add a single legend outside the subplots
    fig.legend(zones_data.keys(), loc='upper center', ncol=1, bbox_to_anchor=(0.8, 0.35), fontsize=16)

    #plt.suptitle('Day-of-Week Variation of Sample Measurement by Year Bin (Zone Average)')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# Main Execution
zone_data = extract_data_by_zone(MD_O3, locations, zones)
binned_zone_data = compute_zone_binned_means(zone_data, year_bins)
plot_binned_zone_means(binned_zone_data)


In [ ]:
binned_zone_data

In [ ]:
# Define zones with corresponding locations
zones = {
    'Upwind': ['Tangerine', 'Coach Line'],
    'Urban Core': ['Craycroft', 'Childrens Park', 'Rose Elementary'],
    'Downwind': ['Saguaro Park', 'Fairgrounds', 'Green Valley']
}

# Define longitude and latitude ranges for each location
locations = {
    'Childrens Park': (-110.9823, 32.29515),
    'Green Valley': (-110.99644, 31.87952),
    'Coach Line': (-111.12716, 32.38082),
    'Rose Elementary': (-110.980134, 32.172995),
    'Fairgrounds': (-110.774357, 32.04767),
    'Tangerine': (-111.06352, 32.425261),
    'Craycroft': (-110.878067, 32.204411),
    'Saguaro Park': (-110.737116, 32.174538)
}

# Extract data and group by zones
def extract_data_by_zone(data, locations, zones):
    zone_data = {zone: [] for zone in zones}  

    for site, (lon, lat) in locations.items():
        data_site = data[(data['Longitude'] == lon) & (data['Latitude'] == lat)].copy()
        data_site['Timestamp'] = pd.to_datetime(data_site['Timestamp'])

        for zone, site_list in zones.items():
            if site in site_list:
                zone_data[zone].append(data_site)

    for zone in zone_data:
        if zone_data[zone]:  
            zone_data[zone] = pd.concat(zone_data[zone])

    return zone_data

# Compute yearly mean or median for each zone
def compute_zone_aggregates(zone_data, method='mean'):
    zone_aggregates = {}

    for zone, df in zone_data.items():
        df['Year'] = df['Timestamp'].dt.year  

        if method == 'mean':
            yearly_agg = df.groupby('Year')['Sample Measurement'].mean().reset_index()
        elif method == 'median':
            yearly_agg = df.groupby('Year')['Sample Measurement'].median().reset_index()
        else:
            raise ValueError("Invalid method. Use 'mean' or 'median'.")

        zone_aggregates[zone] = yearly_agg

    return zone_aggregates

# Compute deltas relative to 2001
def compute_deltas(zone_aggregates):
    deltas = {}

    for zone, df in zone_aggregates.items():
        baseline = df[df['Year'] == 2001]['Sample Measurement'].values

        if len(baseline) == 0:
            print(f"Warning: No data for 2001 in {zone}. Skipping delta computation.")
            continue  

        df['Delta'] = df['Sample Measurement'] - baseline[0]
        deltas[zone] = df

    return deltas

# Plot function for absolute values & deltas
def plot_zonal_time_series(zone_aggregates, deltas):
    fig, axes = plt.subplots(nrows=2, figsize=(14, 10), sharex=True)

    # Absolute values
    for zone, df in zone_aggregates.items():
        axes[0].plot(df['Year'], df['Sample Measurement'], marker='o', linestyle='-', label=zone)

    axes[0].set_ylabel('Mean MDA8 O$_3$ (ppb)', fontsize=14)
    axes[0].set_title('Yearly Mean MDA8 Ozone by Zone', fontsize=16)
    axes[0].tick_params(axis='both', which='major', labelsize=16)
    axes[0].legend(fontsize=12)
    axes[0].grid(True)

    # Delta values (change from 2001)
    for zone, df in deltas.items():
        axes[1].plot(df['Year'], df['Delta'], marker='s', linestyle='--', label=zone)

    axes[1].set_xlabel('Year', fontsize=16)
    axes[1].set_ylabel('Δ MDA8 O$_3$ (ppb) from 2001', fontsize=16)
    axes[1].set_title('Change in MDA8 Ozone Relative to 2001', fontsize=16)
    axes[1].axhline(0, color='black', linewidth=2.5, linestyle='-') 
    axes[1].tick_params(axis='both', which='major', labelsize=16)
    axes[1].legend(fontsize=12)
    axes[1].grid(True)

    # Ensure integer x-axis values
    for ax in axes:
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.show()

# Main Execution
zone_data = extract_data_by_zone(MD_O3, locations, zones)
zone_aggregates = compute_zone_aggregates(zone_data, method='mean')  
deltas = compute_deltas(zone_aggregates)
plot_zonal_time_series(zone_aggregates, deltas)


In [ ]:
MDA_2001 = filtered_df_MDA8[(filtered_df_MDA8['Timestamp'].dt.year >= 2001) & (filtered_df_MDA8['Timestamp'].dt.year <= 2005)]
MDA_2001


In [ ]:
MDA_2006 = filtered_df_MDA8[(filtered_df_MDA8['Timestamp'].dt.year >= 2006) & (filtered_df_MDA8['Timestamp'].dt.year <= 2010)]
MDA_2006


In [ ]:
MDA_2011 = filtered_df_MDA8[(filtered_df_MDA8['Timestamp'].dt.year >= 2011) & (filtered_df_MDA8['Timestamp'].dt.year <= 2015)]
MDA_2011


In [ ]:
MDA_2016 = filtered_df_MDA8[(filtered_df_MDA8['Timestamp'].dt.year >= 2016) & (filtered_df_MDA8['Timestamp'].dt.year <= 2019)]
MDA_2016


In [ ]:
MDA_2022 = MD_O3[(MD_O3['Timestamp'].dt.year >= 2020) & (MD_O3['Timestamp'].dt.year <= 2022)]
MDA_2022


In [ ]:
# Define longitude and latitude ranges for each location
locations = {
    'Childrens Park': (-110.9823, -110.9823, 32.29515, 32.29515),
    'Green Valley': (-110.99644, -110.99644, 31.87952,  31.87952),
    'Coach Line': (-111.12716, -111.12716, 32.38082, 32.38082),
    'Rose Elementary': (-110.980134, -110.980134, 32.172995,  32.172995),
    'Fairgrounds': (-110.774357, -110.774357, 32.04767,  32.04767),
    'Tangerine': (-111.06352, -111.06352, 32.425261,  32.425261),
    'Craycroft': (-110.878067, -110.878067, 32.204411,  32.204411),
    'Saguaro Park': (-110.737116, -110.737116, 32.174538,  32.174538)
}

# Mapping of numeric day values to their corresponding names
#day_mapping = {0: 'Sunday', 1: 'Monday', 2: 'Tuesday', 3: 'Wednesday', 4: 'Thursday', 5: 'Friday', 6: 'Saturday'}
day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
def extract_locationns(data, locations):
    extracted_data = {}
    
    # Iterate through locations
    for location, (min_long, max_long, min_lat, max_lat) in locations.items():
        # Filter data for the location
        data_location = data[(data['Longitude'] >= min_long) & (data['Longitude'] <= max_long) & (data['Latitude'] >= min_lat) & (data['Latitude'] <= max_lat)]
                # Copy 5 columns from the original DataFrame
        data_location = data_location[['Latitude', 'Longitude', 'Timestamp', 'Sample Measurement', 'MDL']].copy()
        
        # Store the extracted data in the dictionary
        extracted_data[location] = data_location
    
    return extracted_data

def average_data_per_day(data):
    # Group data by day of the week and calculate the mean for each day for 'Sample Measurement', 'Latitude', and 'Longitude'
    data['Day'] = data['Timestamp'].dt.day_name()
    data['Day'] = pd.Categorical(data['Day'], categories=day_order, ordered=True)
    daily_avg = data.groupby('Day').agg({
        'Sample Measurement': 'mean',
        'Latitude': 'mean',
        'Longitude': 'mean'
    }).reset_index()

    
    # Rename columns for clarity
    daily_avg.columns = ['Day', 'Sample Measurement', 'Latitude', 'Longitude']
    
    return daily_avg

In [ ]:
# Extract data for each location
extracted_data01 = extract_locationns(MDA_2001, locations)
extracted_data06 = extract_locationns(MDA_2006, locations)
extracted_data11 = extract_locationns(MDA_2011, locations)
extracted_data16 = extract_locationns(MDA_2016, locations)
extracted_data22 = extract_locationns(MDA_2022, locations)
MDA_2022
# Calculate average per day for each location
hourly_avg = {}
for location, data_location in extracted_data01.items():
    hourly_avg[location] = average_data_per_day(data_location)
hourly_avg06 = {}
for location, data_location in extracted_data06.items():
    hourly_avg06[location] = average_data_per_day(data_location)
hourly_avg11 = {}
for location, data_location in extracted_data11.items():
    hourly_avg11[location] = average_data_per_day(data_location)
hourly_avg16 = {}
for location, data_location in extracted_data16.items():
    hourly_avg16[location] = average_data_per_day(data_location)
hourly_avg22 = {}
for location, data_location in extracted_data22.items():
    hourly_avg22[location] = average_data_per_day(data_location)

In [ ]:
def extract_locations(df, locations, seasons):
    extracted_dfs = {}
    
    for location, (long_min, long_max, lat_min, lat_max) in locations.items():
        # Filter rows based on longitude and latitude range for the current location
        location_df = df[(df['Longitude'] >= long_min) & (df['Longitude'] <= long_max) &
                         (df['Latitude'] >= lat_min) & (df['Latitude'] <= lat_max)].copy()
        # Copy 5 columns from the original DataFrame
        location_df = location_df[['Latitude', 'Longitude', 'Timestamp', 'Sample Measurement', 'MDL']].copy()    
        # Convert 'Timestamp' column to datetime if it's not already
        location_df['Timestamp'] = pd.to_datetime(location_df['Timestamp'])
        
        # Set 'Timestamp' column as the index
        location_df.set_index('Timestamp', inplace=True)
        
        # Filter rows based on the selected seasons for the current location
        season_df = location_df[location_df.index.month.isin(seasons)].copy()
        
        # Group hourly data into daily data using the maximum value in 'Sample Measurement' column
        season_df = season_df.groupby(pd.Grouper(freq='D')).agg(
            {col: 'mean' if col in ['Sample Measurement', 'MDL'] else 'last' for col in season_df.columns})
        
        # Extract the day of the week from the index and map it to day name
        season_df['Day_of_Week'] = season_df.index.day_name()
        
        # Group by day of the week and take the mean
        mean_by_day_of_week = season_df.groupby('Day_of_Week').mean()
       # Categorize and sort days of the week
        day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
        mean_by_day_of_week = mean_by_day_of_week.reindex(day_order)
        
        # Store the extracted DataFrame in a dictionary
        extracted_dfs[location] = mean_by_day_of_week
    
    return extracted_dfs

# Define longitude and latitude ranges for each location
locations = {
    'Childrens Park': (-110.9823, -110.9823, 32.29515, 32.29515),
    'Green Valley': (-110.99644, -110.99644, 31.87952,  31.87952),
    'Coach Line': (-111.12716, -111.12716, 32.38082, 32.38082),
    'Rose Elementary': (-110.980134, -110.980134, 32.172995,  32.172995),
    'Fairgrounds': (-110.774357, -110.774357, 32.04767,  32.04767),
    'Tangerine': (-111.06352, -111.06352, 32.425261,  32.425261),
    'Craycroft': (-110.878067, -110.878067, 32.204411,  32.204411),
    'Saguaro Park': (-110.737116, -110.737116, 32.174538,  32.174538)
}

# Define seasons
winter_season = [12, 1, 2]
spring_season = [3, 4, 5]
summer_season_dry = [6]
summer_season = [7, 8]
fall_season = [9, 10, 11]


In [ ]:
# Extract data for the summer season
# Call the function to extract locations based on seasons
winter_dataOzoneMD01 = extract_locations(MDA_2001, locations, winter_season)
spring_dataOzoneMD01 = extract_locations(MDA_2001, locations, spring_season)
summer_data_dryOzoneMD01 = extract_locations(MDA_2001, locations, summer_season_dry)
summer_dataOzoneMD01 = extract_locations(MDA_2001, locations, summer_season)
fall_dataOzoneMD01 = extract_locations(MDA_2001, locations, fall_season)

In [ ]:
# Extract data for the summer season
# Call the function to extract locations based on seasons
winter_dataOzoneMD06 = extract_locations(MDA_2006, locations, winter_season)
spring_dataOzoneMD06 = extract_locations(MDA_2006, locations, spring_season)
summer_data_dryOzoneMD06 = extract_locations(MDA_2006, locations, summer_season_dry)
summer_dataOzoneMD06 = extract_locations(MDA_2006, locations, summer_season)
fall_dataOzoneMD06 = extract_locations(MDA_2006, locations, fall_season)

In [ ]:
# Extract data for the summer season
# Call the function to extract locations based on seasons
winter_dataOzoneMD11 = extract_locations(MDA_2011, locations, winter_season)
spring_dataOzoneMD11 = extract_locations(MDA_2011, locations, spring_season)
summer_data_dryOzoneMD11 = extract_locations(MDA_2011, locations, summer_season_dry)
summer_dataOzoneMD11 = extract_locations(MDA_2011, locations, summer_season)
fall_dataOzoneMD11 = extract_locations(MDA_2011, locations, fall_season)

In [ ]:
# Call the function to extract locations based on seasons
winter_dataOzoneMD = extract_locations(MDA_2016, locations, winter_season)
spring_dataOzoneMD = extract_locations(MDA_2016, locations, spring_season)
summer_data_dryOzoneMD = extract_locations(MDA_2016, locations, summer_season_dry)
summer_dataOzoneMD = extract_locations(MDA_2016, locations, summer_season)
fall_dataOzoneMD = extract_locations(MDA_2016, locations, fall_season)

In [ ]:
# Call the function to extract locations based on seasons
winter_dataOzone22 = extract_locations(MDA_2022, locations, winter_season)
spring_dataOzone22 = extract_locations(MDA_2022, locations, spring_season)
summer_data_dryOzone22 = extract_locations(MDA_2022, locations, summer_season_dry)
summer_dataOzone22 = extract_locations(MDA_2022, locations, summer_season)
fall_dataOzone22 = extract_locations(MDA_2022, locations, fall_season)

In [ ]:
from scipy.stats import ttest_ind
import pandas as pd

# Function to calculate percentage change
def calculate_percent_change(weekday_value, weekend_value):
    if weekday_value == 0:  # Avoid division by zero
        return 0
    percent_change = ((weekend_value - weekday_value) / weekday_value) * 100
    return percent_change

# Perform t-test for specified locations
def perform_ttest_for_locations(locations, season_data):
    results = {}
    for location in locations:
        # Extract the DataFrame for the specific location
        location_data = season_data.get(location)
        
        if location_data is not None and not location_data.empty:
            # Ensure the 'Sample Measurement' column is numeric
            location_data['Sample Measurement'] = pd.to_numeric(location_data['Sample Measurement'], errors='coerce')

            # Separate the data into weekday and weekend samples based on 'Day_of_Week'
            weekday_data = location_data[location_data.index.isin(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'])]
            weekend_data = location_data[location_data.index.isin(['Saturday', 'Sunday'])]

            if not weekday_data.empty and not weekend_data.empty:
                # Perform t-test
                t_statistic, p_value = ttest_ind(
                    weekend_data['Sample Measurement'], 
                    weekday_data['Sample Measurement'], 
                    equal_var=False, 
                    nan_policy='omit'
                )

                # Calculate mean values for weekday and weekend samples
                mean_weekday = weekday_data['Sample Measurement'].mean()
                mean_weekend = weekend_data['Sample Measurement'].mean()

                # Calculate percentage change and absolute difference
                percent_change = calculate_percent_change(mean_weekday, mean_weekend)
                abs_difference = mean_weekend - mean_weekday

                # Calculate mean latitude and longitude
                mean_latitude = location_data['Latitude'].mean()
                mean_longitude = location_data['Longitude'].mean()

                # Store the results
                results[location] = {
                    'T-statistic': t_statistic,
                    'P-value': p_value,
                    'Percent Change': percent_change,
                    'Abs Difference': abs_difference,
                    'Latitude': mean_latitude,
                    'Longitude': mean_longitude,
                    'weekday': mean_weekday,
                    'weekend': mean_weekend
                }
    return results

locations_of_interest = ['Childrens Park', 'Green Valley', 'Coach Line', 'Rose Elementary', 'Fairgrounds', 'Tangerine', 
                         'Craycroft', 'Saguaro Park']


In [ ]:
# Assuming `hour_19_spring_season`, `hour_20_spring_season`, and `hour_22_spring_season` are already defined and available

# Perform the t-test for the specified locations for hour_19_winter_season
t_test_resultwinter_season01 = perform_ttest_for_locations(locations_of_interest, winter_dataOzoneMD01)
# Perform the t-test for the specified locations for hour_19_winter_season
t_test_resultwinter_season06 = perform_ttest_for_locations(locations_of_interest, winter_dataOzoneMD06)
# Perform the t-test for the specified locations for hour_19_winter_season
t_test_resultwinter_season11 = perform_ttest_for_locations(locations_of_interest, winter_dataOzoneMD11)
t_test_resultwinter_season22 = perform_ttest_for_locations(locations_of_interest, winter_dataOzone22)
# Perform the t-test for the specified locations for hour_20_winter_season
t_test_resultwinter_season20 = perform_ttest_for_locations(locations_of_interest, winter_dataOzoneMD)

t_test_resultspring_season22 = perform_ttest_for_locations(locations_of_interest, spring_dataOzone22)
# Perform the t-test for the specified locations for hour_19_spring_season
t_test_resultspring_season20 = perform_ttest_for_locations(locations_of_interest, spring_dataOzoneMD)
# Perform the t-test for the specified locations for hour_19_spring_season
t_test_resultspring_season11 = perform_ttest_for_locations(locations_of_interest, spring_dataOzoneMD11)
# Perform the t-test for the specified locations for hour_19_spring_season
t_test_resultspring_season06 = perform_ttest_for_locations(locations_of_interest, spring_dataOzoneMD06)

# Perform the t-test for the specified locations for hour_20_spring_season
t_test_resultspring_season01 = perform_ttest_for_locations(locations_of_interest, spring_dataOzoneMD01)

t_test_resultsummer_season22 = perform_ttest_for_locations(locations_of_interest, summer_dataOzone22)
t_test_resultsummer_season20 = perform_ttest_for_locations(locations_of_interest, summer_dataOzoneMD)
# Perform the t-test for the specified locations for hour_19_summer_season
t_test_resultsummer_season11 = perform_ttest_for_locations(locations_of_interest, summer_dataOzoneMD11)
# Perform the t-test for the specified locations for hour_19_summer_season
t_test_resultsummer_season06 = perform_ttest_for_locations(locations_of_interest, summer_dataOzoneMD06)

# Perform the t-test for the specified locations for hour_20_summer_season
t_test_resultsummer_season01 = perform_ttest_for_locations(locations_of_interest, summer_dataOzoneMD01)


t_test_resultsummer_season_dry11 = perform_ttest_for_locations(locations_of_interest, summer_data_dryOzoneMD11)
# Perform the t-test for the specified locations for hour_19_summer_season_dry
t_test_resultsummer_season_dry06 = perform_ttest_for_locations(locations_of_interest, summer_data_dryOzoneMD06)
# Perform the t-test for the specified locations for hour_19_summer_season_dry
t_test_resultsummer_season_dry01 = perform_ttest_for_locations(locations_of_interest, summer_data_dryOzoneMD01)

# Perform the t-test for the specified locations for hour_20_summer_season_dry
t_test_resultsummer_season_dry20 = perform_ttest_for_locations(locations_of_interest, summer_data_dryOzoneMD)
t_test_resultsummer_season_dry22 = perform_ttest_for_locations(locations_of_interest, summer_data_dryOzone22)

t_test_resultfall_season22 = perform_ttest_for_locations(locations_of_interest, fall_dataOzone22)
t_test_resultfall_season20 = perform_ttest_for_locations(locations_of_interest, fall_dataOzoneMD)
# Perform the t-test for the specified locations for hour_19_fall_season
t_test_resultfall_season11 = perform_ttest_for_locations(locations_of_interest, fall_dataOzoneMD11)
# Perform the t-test for the specified locations for hour_19_fall_season
t_test_resultfall_season06 = perform_ttest_for_locations(locations_of_interest, fall_dataOzoneMD06)

# Perform the t-test for the specified locations for hour_20_fall_season
t_test_resultfall_season01 = perform_ttest_for_locations(locations_of_interest, fall_dataOzoneMD01)



# Function to print results
def print_results(t_test_results, label):
    print(f"Results for {label}:")
    for location, result in t_test_results.items():
        print(f"Location: {location}")
        print(f"T-statistic: {result['T-statistic']}")
        print(f"P-value: {result['P-value']}")
        print(f"Percent Change: {result['Percent Change']}%")
        print(f"Latitude: {result['Latitude']}")
        print(f"Longitude: {result['Longitude']}")
        print()
print_results(t_test_resultwinter_season11, 'winter_season 11')
print_results(t_test_resultwinter_season06, 'winter_season 06')
print_results(t_test_resultwinter_season01, 'winter_season 01')
print_results(t_test_resultwinter_season20, 'winter_season 20')

# Print results for each dataset
print_results(t_test_resultspring_season11, 'spring_season 11')
print_results(t_test_resultspring_season06, 'spring_season 06')
print_results(t_test_resultspring_season01, 'spring_season 01')
print_results(t_test_resultspring_season20, 'spring_season 20')

print_results(t_test_resultsummer_season11, 'summer_season 11')
print_results(t_test_resultsummer_season06, 'summer_season 06')
print_results(t_test_resultsummer_season01, 'summer_season 01')
print_results(t_test_resultsummer_season20, 'summer_season 20')

print_results(t_test_resultsummer_season_dry11, 'summer_season_dry 11')
print_results(t_test_resultsummer_season_dry06, 'summer_season_dry 06')
print_results(t_test_resultsummer_season_dry01, 'summer_season_dry 01')
print_results(t_test_resultsummer_season_dry20, 'summer_season_dry 20')

# Define the lists to store the data for each hour
def store_results(t_test_results):
    locations = []
    t_statistics = []
    p_values = []
    percent_changes = []
    Difference = []
    Longitude = []
    Latitude = []
    weekday = []
    weekend = []

    for location, result in t_test_results.items():
        locations.append(location)
        t_statistics.append(result['T-statistic'])
        p_values.append(result['P-value'])
        percent_changes.append(result['Percent Change'])
        Latitude.append(result['Latitude'])
        Longitude.append(result['Longitude'])
        Difference.append(result['Abs Difference'])
        weekday.append(result['weekday'])
        weekend.append(result['weekend'])

    return pd.DataFrame({
        'Location': locations,
        'T-statistic': t_statistics,
        'P-value': p_values,
        'Percent Change': percent_changes,
        'Difference': Difference,
        'Latitude': Latitude,
        'Longitude': Longitude,
        'weekday': weekday,
        'weekend': weekend
    })

# Create DataFrames for each hour
winter_season11 = store_results(t_test_resultwinter_season11)
winter_season06 = store_results(t_test_resultwinter_season06)
winter_season01 = store_results(t_test_resultwinter_season01)
winter_season20 = store_results(t_test_resultwinter_season20)
winter_season22 = store_results(t_test_resultwinter_season22)

# Create DataFrames for each hour
spring_season11 = store_results(t_test_resultspring_season11)
spring_season06 = store_results(t_test_resultspring_season06)
spring_season01 = store_results(t_test_resultspring_season01)
spring_season20 = store_results(t_test_resultspring_season20)
spring_season22 = store_results(t_test_resultspring_season22)

summer_season11 = store_results(t_test_resultsummer_season11)
summer_season06 = store_results(t_test_resultsummer_season06)
summer_season01 = store_results(t_test_resultsummer_season01)
summer_season20 = store_results(t_test_resultsummer_season20)
summer_season22 = store_results(t_test_resultsummer_season22)

summer_season_dry06 = store_results(t_test_resultsummer_season_dry06)
summer_season_dry01 = store_results(t_test_resultsummer_season_dry01)
summer_season_dry11 = store_results(t_test_resultsummer_season_dry11)
summer_season_dry20 = store_results(t_test_resultsummer_season_dry20)
summer_season_dry22 = store_results(t_test_resultsummer_season_dry22)

fall_season11 = store_results(t_test_resultfall_season11)
fall_season06 = store_results(t_test_resultfall_season06)
fall_season01 = store_results(t_test_resultfall_season01)
fall_season20 = store_results(t_test_resultfall_season20)
fall_season22 = store_results(t_test_resultfall_season22)

# Print the DataFrames
print(winter_season20)
print(spring_season01)
print(spring_season11)
print(spring_season20)


In [ ]:
summer_season_dry22

In [ ]:
from scipy.stats import ttest_ind

# Calculate percentage change from weekday to weekend
def calculate_percent_changee(weekday_value, weekend_value):
    percent_change = ((weekend_value - weekday_value) / weekday_value) * 100
    return percent_change

# Perform t-test for the given locations
def perform_ttest_for_locationns(hourly_avg):
    results = {}
    for location, location_data in hourly_avg.items():
        if location_data is not None and not location_data.empty:
            # Separate the data into weekday and weekend samples based on the 'Day' column
            weekday_data = location_data[location_data['Day'].isin(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'])]
            weekend_data = location_data[location_data['Day'].isin(['Saturday', 'Sunday'])]
            
            if not weekday_data.empty and not weekend_data.empty:
                # Perform t-test using sample measurements for weekdays and weekends
                t_statistic, p_value = ttest_ind(weekend_data['Sample Measurement'], weekday_data['Sample Measurement'], equal_var=False)
                
                # Calculate mean values for weekday and weekend samples
                mean_weekday = weekday_data['Sample Measurement'].mean()
                mean_weekend = weekend_data['Sample Measurement'].mean()
                
                # Calculate percentage change from weekday to weekend
                percent_change = calculate_percent_changee(mean_weekday, mean_weekend)

                abs_difference = mean_weekend - mean_weekday
                
                # Extract the first value of latitude and longitude
                Latitude = location_data['Latitude'].iloc[0]
                Longitude = location_data['Longitude'].iloc[0]
                
                # Store the results for this location
                results[location] = {
                    'T-statistic': t_statistic, 
                    'P-value': p_value, 
                    'Percent Change': percent_change,
                    'Abs Difference':  abs_difference,
                    'Latitude': Latitude, 
                    'Longitude': Longitude,
                    'weekday': mean_weekday, 
                    'weekend': mean_weekend
                }
           
    
    return results


In [ ]:
t_test_resultsfull = perform_ttest_for_locationns(hourly_avg)
# Print the results
for location, result in t_test_resultsfull.items():
    print(f"Location: {location}")
    print(f"T-statistic: {result['T-statistic']}")
    print(f"P-value: {result['P-value']}")
    print(f"Percent Change: {result['Percent Change']}%")
    print(f"Latitude: {result['Latitude']}")
    print(f"Longitude: {result['Longitude']}")
    print()

# Define the lists to store the data
locations = []
t_statistics = []
p_values = []
percent_changes = []
Difference = []
Longitude = []
Latitude = []
weekday = []
weekend = []
# Iterate through the results and append the data to the lists
for location, result in t_test_resultsfull.items():
    locations.append(location)
    t_statistics.append(result['T-statistic'])
    p_values.append(result['P-value'])
    percent_changes.append(result['Percent Change'])
    Latitude.append(result['Latitude'])
    Longitude.append(result['Longitude'])
    Difference.append(result['Abs Difference'])
    weekday.append(result['weekday'])
    weekend.append(result['weekend'])

# Create a DataFrame from the lists
Total_01 = pd.DataFrame({
    'Location': locations,
    'T-statistic': t_statistics,
    'P-value': p_values,
    'Percent Change': percent_changes,
    'Difference': Difference,
    'Latitude': Latitude,
    'Longitude': Longitude,
    'weekday': weekday,
    'weekend':weekend
})

# Print the DataFrame
print(Total_01)

In [ ]:
t_test_resultsf06 = perform_ttest_for_locationns(hourly_avg06)
# Print the results
for location, result in t_test_resultsf06.items():
    print(f"Location: {location}")
    print(f"T-statistic: {result['T-statistic']}")
    print(f"P-value: {result['P-value']}")
    print(f"Percent Change: {result['Percent Change']}%")
    print(f"Latitude: {result['Latitude']}")
    print(f"Longitude: {result['Longitude']}")
    print()

# Define the lists to store the data
locations = []
t_statistics = []
p_values = []
percent_changes = []
Difference = []
Longitude = []
Latitude = []
weekday = []
weekend = []
# Iterate through the results and append the data to the lists
for location, result in t_test_resultsf06.items():
    locations.append(location)
    t_statistics.append(result['T-statistic'])
    p_values.append(result['P-value'])
    percent_changes.append(result['Percent Change'])
    Latitude.append(result['Latitude'])
    Longitude.append(result['Longitude'])
    Difference.append(result['Abs Difference'])
    weekday.append(result['weekday'])
    weekend.append(result['weekend'])

# Create a DataFrame from the lists
Total_06 = pd.DataFrame({
    'Location': locations,
    'T-statistic': t_statistics,
    'P-value': p_values,
    'Percent Change': percent_changes,
    'Difference': Difference,
    'Latitude': Latitude,
    'Longitude': Longitude,
    'weekday': weekday,
    'weekend':weekend
})


# Print the DataFrame
print(Total_06)

In [ ]:
t_test_resultsf11 = perform_ttest_for_locationns(hourly_avg11)
# Print the results
for location, result in t_test_resultsf11.items():
    print(f"Location: {location}")
    print(f"T-statistic: {result['T-statistic']}")
    print(f"P-value: {result['P-value']}")
    print(f"Percent Change: {result['Percent Change']}%")
    print(f"Latitude: {result['Latitude']}")
    print(f"Longitude: {result['Longitude']}")
    print()

# Define the lists to store the data
locations = []
t_statistics = []
p_values = []
percent_changes = []
Difference = []
Longitude = []
Latitude = []
weekday = []
weekend = []
# Iterate through the results and append the data to the lists
for location, result in t_test_resultsf11.items():
    locations.append(location)
    t_statistics.append(result['T-statistic'])
    p_values.append(result['P-value'])
    percent_changes.append(result['Percent Change'])
    Latitude.append(result['Latitude'])
    Longitude.append(result['Longitude'])
    Difference.append(result['Abs Difference'])
    weekday.append(result['weekday'])
    weekend.append(result['weekend'])

# Create a DataFrame from the lists
Total_11 = pd.DataFrame({
    'Location': locations,
    'T-statistic': t_statistics,
    'P-value': p_values,
    'Percent Change': percent_changes,
    'Difference': Difference,
    'Latitude': Latitude,
    'Longitude': Longitude,
    'weekday': weekday,
    'weekend':weekend
})


# Print the DataFrame
print(Total_11)

In [ ]:
t_test_resultsf16 = perform_ttest_for_locationns(hourly_avg16)
# Print the results
for location, result in t_test_resultsf16.items():
    print(f"Location: {location}")
    print(f"T-statistic: {result['T-statistic']}")
    print(f"P-value: {result['P-value']}")
    print(f"Percent Change: {result['Percent Change']}%")
    print(f"Latitude: {result['Latitude']}")
    print(f"Longitude: {result['Longitude']}")
    print()

# Define the lists to store the data
locations = []
t_statistics = []
p_values = []
percent_changes = []
Difference = []
Longitude = []
Latitude = []
weekday = []
weekend = []
# Iterate through the results and append the data to the lists
for location, result in t_test_resultsf16.items():
    locations.append(location)
    t_statistics.append(result['T-statistic'])
    p_values.append(result['P-value'])
    percent_changes.append(result['Percent Change'])
    Latitude.append(result['Latitude'])
    Longitude.append(result['Longitude'])
    Difference.append(result['Abs Difference'])
    weekday.append(result['weekday'])
    weekend.append(result['weekend'])

# Create a DataFrame from the lists
Total_16 = pd.DataFrame({
    'Location': locations,
    'T-statistic': t_statistics,
    'P-value': p_values,
    'Percent Change': percent_changes,
    'Difference': Difference,
    'Latitude': Latitude,
    'Longitude': Longitude,
    'weekday': weekday,
    'weekend':weekend
})

# Print the DataFrame
print(Total_16)

In [ ]:
t_test_resultsf22 = perform_ttest_for_locationns(hourly_avg22)
# Print the results
for location, result in t_test_resultsf22.items():
    print(f"Location: {location}")
    print(f"T-statistic: {result['T-statistic']}")
    print(f"P-value: {result['P-value']}")
    print(f"Percent Change: {result['Percent Change']}%")
    print(f"Latitude: {result['Latitude']}")
    print(f"Longitude: {result['Longitude']}")
    print()

# Define the lists to store the data
locations = []
t_statistics = []
p_values = []
percent_changes = []
Difference = []
Longitude = []
Latitude = []
weekday = []
weekend = []
# Iterate through the results and append the data to the lists
for location, result in t_test_resultsf22.items():
    locations.append(location)
    t_statistics.append(result['T-statistic'])
    p_values.append(result['P-value'])
    percent_changes.append(result['Percent Change'])
    Latitude.append(result['Latitude'])
    Longitude.append(result['Longitude'])
    Difference.append(result['Abs Difference'])
    weekday.append(result['weekday'])
    weekend.append(result['weekend'])

# Create a DataFrame from the lists
Total_22 = pd.DataFrame({
    'Location': locations,
    'T-statistic': t_statistics,
    'P-value': p_values,
    'Percent Change': percent_changes,
    'Difference': Difference,
    'Latitude': Latitude,
    'Longitude': Longitude,
    'weekday': weekday,
    'weekend':weekend
})

# Print the DataFrame
print(Total_22)

In [ ]:
# Function to calculate percentage change between two years
def calculate_percentage(df_new, df_old, suffix_new, suffix_old):
    # Merge DataFrames on Location
    aligned_df = pd.merge(df_new, df_old, on='Location', suffixes=(suffix_new, suffix_old))

    # Calculate percentage difference for weekend, weekday, and difference
    aligned_df[f'weekend_pct_diff{suffix_new}{suffix_old}'] = (
        (aligned_df[f'weekend{suffix_new}'] - aligned_df[f'weekend{suffix_old}']) 
        / aligned_df[f'weekend{suffix_old}']) * 100
    aligned_df[f'weekday_pct_diff{suffix_new}{suffix_old}'] = (
        (aligned_df[f'weekday{suffix_new}'] - aligned_df[f'weekday{suffix_old}']) 
        / aligned_df[f'weekday{suffix_old}']) * 100
    aligned_df[f'difference_pct_diff{suffix_new}{suffix_old}'] = (
        (aligned_df[f'Difference{suffix_new}'] - aligned_df[f'Difference{suffix_old}']) 
        / aligned_df[f'Difference{suffix_old}']) * 100

    # Add Longitude and Latitude from the original DataFrame
    aligned_df['Longitude'] = df_new['Longitude']
    aligned_df['Latitude'] = df_new['Latitude']

    return aligned_df[['Location', 'Longitude', 'Latitude', 
                       f'weekend_pct_diff{suffix_new}{suffix_old}', 
                       f'weekday_pct_diff{suffix_new}{suffix_old}', 
                       f'difference_pct_diff{suffix_new}{suffix_old}']]

# Assuming the DataFrames for each year are already defined
# Extract the DataFrames for each year
df_01 = Total_01
df_06 = Total_06
df_11 = Total_11
df_20 = Total_16
df_22 = Total_22

# Calculate percentage changes and assign to separate DataFrames
total_pct_change_06_01 = calculate_percentage(df_06, df_01, '_06', '_01')
total_pct_change_11_01 = calculate_percentage(df_11, df_01, '_11', '_01')
total_pct_change_16_01 = calculate_percentage(df_20, df_01, '_16', '_01')
total_pct_change_22_01 = calculate_percentage(df_22, df_01, '_22', '_01')


In [ ]:
df_22

In [ ]:
# Define the seasons and corresponding DataFrames
seasons = {
    'winter': ['winter_season01', 'winter_season06', 'winter_season11', 'winter_season20', 'winter_season22'],
    'spring': ['spring_season01', 'spring_season06', 'spring_season11', 'spring_season20', 'spring_season22'],
    'sum_dry': ['summer_season_dry01', 'summer_season_dry06', 'summer_season_dry11', 'summer_season_dry20', 'summer_season_dry22'],
    'summer': ['summer_season01', 'summer_season06', 'summer_season11', 'summer_season20', 'summer_season22'],
    'fall': ['fall_season01', 'fall_season06', 'fall_season11', 'fall_season20', 'fall_season22']
}

# Function to calculate percentage change between two years
def calculate_percentage_change(df_new, df_old, suffix_new, suffix_old):
    # Merge DataFrames on Location
    aligned_df = pd.merge(df_new, df_old, on='Location', suffixes=(suffix_new, suffix_old))

    # Calculate percentage difference for weekend and weekdays
    aligned_df[f'weekend_pct_diff{suffix_new}{suffix_old}'] = ((aligned_df[f'weekend{suffix_new}'] - aligned_df[f'weekend{suffix_old}']) / aligned_df[f'weekend{suffix_old}']) * 100
    aligned_df[f'weekday_pct_diff{suffix_new}{suffix_old}'] = ((aligned_df[f'weekday{suffix_new}'] - aligned_df[f'weekday{suffix_old}']) / aligned_df[f'weekday{suffix_old}']) * 100
    aligned_df[f'difference_pct_diff{suffix_new}{suffix_old}'] = ((aligned_df[f'Difference{suffix_new}'] - aligned_df[f'Difference{suffix_old}']) / aligned_df[f'Difference{suffix_old}']) * 100

    # Add Longitude and Latitude from the original DataFrame
    aligned_df['Longitude'] = df_new['Longitude']
    aligned_df['Latitude'] = df_new['Latitude']

    return aligned_df[['Location', 'Longitude', 'Latitude', f'weekend_pct_diff{suffix_new}{suffix_old}', f'weekday_pct_diff{suffix_new}{suffix_old}', f'difference_pct_diff{suffix_new}{suffix_old}']]

# Iterate over each season and calculate the percentage change for 05-97, 11-05, and 17-11
for season, dataframes in seasons.items():
    ddf_01, ddf_06, ddf_11, ddf_20, ddf_22 = [globals()[df_name] for df_name in dataframes]

    # Calculate percentage changes and assign to separate DataFrames
    globals()[f'{season}_pct_change_06_01'] = calculate_percentage_change(ddf_06, ddf_01, '_06', '_01')
    globals()[f'{season}_pct_change_11_01'] = calculate_percentage_change(ddf_11, ddf_01, '_11', '_01')
    globals()[f'{season}_pct_change_20_01'] = calculate_percentage_change(ddf_20, ddf_01, '_20', '_01')
    globals()[f'{season}_pct_change_22_01'] = calculate_percentage_change(ddf_22, ddf_01, '_22', '_01')

# Example: Display the results for Spring 05-97

# Similar saving for other seasons


In [ ]:
import geopandas as gpd
Pimaa = "C:/Users/taiwoajayi/OneDrive - University of Arizona/Arizona_ozone/Ozone/tl_2023_04019_roads.shp"
# Load the shapefiles using geopandas
gdf_pima = gpd.read_file(Pimaa)

In [ ]:
titles = ['2001-2005', '2006-2010', '2011-2015', '2016-2019', '2020-2022']

# Assume percentage change DataFrames have been calculated as:
# total_pct_change_05_97, total_pct_change_11_05, total_pct_change_17_11
pct_changes = [total_pct_change_06_01, total_pct_change_11_01, total_pct_change_16_01, total_pct_change_22_01]
dataframes = [Total_01, Total_06, Total_11, Total_16, Total_22]

# Normalize the colormap to have the middle at 0
all_percent_changes = pd.concat([df['Difference'] for df in dataframes])
vmin = np.floor(all_percent_changes.min())
vmax = np.ceil(all_percent_changes.max())

if vmin >= 0:
    vmin = -1

norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

# List of dataframes
#dataframes = [Total_01, Total_06, Total_11, Total_16]

# Create the subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw={'projection': ccrs.PlateCarree()}, dpi=300)
axes = axes.flatten()
axes[-1].set_visible(False)

plt.subplots_adjust(wspace=0.05, hspace=0.25)

# Predefined suffixes for percentage change columns
suffixes = ['06_01', '11_01', '16_01', '22_01']

# Collect all percentage changes for scaling the marker size in the legend
all_pct_changes = []

# Plot each dataframe in a subplot
for i, (ax, df, title) in enumerate(zip(axes, dataframes, titles)):
    ax.set_extent([-111.3, -110.50, 31.75, 32.55], crs=ccrs.PlateCarree())  # Adjust the extent as needed
    ax.coastlines()

    # Add additional map features, such as rivers, borders, or land color
    ax.add_feature(cfeature.RIVERS)
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.LAND, facecolor='white')

    # Plot the shapefiles
    for gdf, color in zip([gdf_pima], ['gray']):
        ax.add_geometries(gdf.geometry, crs=ccrs.PlateCarree(), edgecolor=color, facecolor='none', linewidth=1)

    # Define the marker style for the first subplot (fixed size and up triangles)

    if i == 0:
        scatter = ax.scatter(df['Longitude'], df['Latitude'], c=df['Difference'], cmap='coolwarm', norm=norm,
                             edgecolors='k', zorder=35, s=500, marker='^')
    
    # For the remaining subplots, use different marker styles based on the change direction and scaled sizes
    else:
        pct_df = pct_changes[i-1]  # Get the corresponding percentage change DataFrame

        # Use predefined suffixes to match column names
        suffix = suffixes[i-1]

        for _, row in pct_df.iterrows():
            longitude = row['Longitude']
            latitude = row['Latitude']
            difference = df.loc[df['Location'] == row['Location'], 'Difference'].values[0]
            difference_pct_change = row[f'difference_pct_diff_{suffix}']

            # Append percentage changes for legend scaling
            all_pct_changes.append(np.abs(difference_pct_change))

            # Marker type: up triangle for positive change, down triangle for negative
            marker = '^' if difference_pct_change > 0 else 'v'

            # Plot with scaled marker size
            max_marker_size = 700  # Set a maximum limit for marker size
            scaling_factor = 20 
            marker_size = np.clip(np.abs(difference_pct_change) * scaling_factor, 60, max_marker_size) # Adjust scaling factor
            
            scatter = ax.scatter(longitude, latitude, c=[difference], cmap='seismic', norm=norm, edgecolors='k',
                                 s=marker_size, marker=marker, zorder=35)
                            #size = abs(diff_pct_change) * 100  # Adjust scaling factor as necessary



    # Add a title
    ax.set_title(title, fontsize=20)

    # Add state borders
    ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='black')

    for ax in axes[:-1]:
        # Add gridlines
        gl = ax.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
        gl.xlabels_top = False
        gl.ylabels_right = False
        gl.xlabel_style = {'size': 18}
        gl.ylabel_style = {'size': 18}

        # Hide labels on the right and top sides of the subplot
        gl.xlabels_top = False
        gl.ylabels_right = False

    '''for i in range(2):
        for j in range(2):
            axx = axes[i * 2 + j]  # Indexing as a 1D array
            gl = axx.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
            gl.xlabels_top = False
            gl.ylabels_right = False
            gl.xlabel_style = {'size': 18}
            gl.ylabel_style = {'size': 18}

            if i != 1:  
                gl.xlabels_bottom = False
            if j != 0:  
                gl.ylabels_left = False'''

    # Set gridlines
    #gl = ax.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
    #gl.xlabels_top = False
    #gl.ylabels_right = False
    #gl.xlabel_style = {'size': 14}
    #gl.ylabel_style = {'size': 14}

# Add a single colorbar for all subplots
cbar = fig.colorbar(scatter, ax=axes, orientation='horizontal', pad=0.05, extend='both', fraction=0.05, aspect=30)
cbar.ax.tick_params(labelsize=20)
cbar.set_label('MDA8 O$_3$ Weekend-Weekday Difference (ppb)', fontsize=20)

# Calculate data-based percentage ranges using quantiles
low_range = (np.percentile(all_pct_changes, 0), np.percentile(all_pct_changes, 33))  # 0-33%
mid_range = (np.percentile(all_pct_changes, 34), np.percentile(all_pct_changes, 66))  # 34-66%
high_range = (np.percentile(all_pct_changes, 67), np.percentile(all_pct_changes, 100))  # 67-100%

# Custom legend for marker size (absolute percentage ranges)
legend_elements = [
    mlines.Line2D([], [], marker='^', color='w', label=f'{low_range[0]:.0f}% - {low_range[1]:.0f}%', markersize=low_range[1] * 0.2,
                  markerfacecolor='gray', markeredgewidth=2),
    mlines.Line2D([], [], marker='^', color='w', label=f'{mid_range[0]:.0f}% - {mid_range[1]:.0f}%', markersize=mid_range[1] * 0.3,
                  markerfacecolor='gray', markeredgewidth=2),
    mlines.Line2D([], [], marker='^', color='w', label=f'{high_range[0]:.0f}% +', markersize=high_range[1] * 0.2,
                  markerfacecolor='gray', markeredgewidth=2),
]

# Add custom legend for marker size, adjust position to avoid overlap
#plt.legend(handles=legend_elements, title="Percent Change (Absolute)", title_fontsize = 14, labelspacing=1.0, handletextpad=0.2, loc='lower center', bbox_to_anchor=(0.72, 1.82), ncol=1, fontsize=14)
#fig.tight_layout()
fig.legend(handles=legend_elements, title="Percent Change (Absolute)", title_fontsize=18, labelspacing=1.0, handletextpad=0.2,
           loc='center', bbox_to_anchor=(0.75, 0.35), fontsize=18, ncol=1, frameon=False)
plt.show()


In [ ]:
# List of dataframes
titles = ['2001-2005', '2006-2010', '2011-2015', '2016-2019', '2020-2022']

# Assume percentage change DataFrames have been calculated as:
# total_pct_change_05_97, total_pct_change_11_05, total_pct_change_17_11
pct_changes = [total_pct_change_06_01, total_pct_change_11_01, total_pct_change_16_01, total_pct_change_22_01]
dataframes = [Total_01, Total_06, Total_11, Total_16, Total_22]

# Normalize the colormap to have the middle at 0
all_percent_changes = pd.concat([df['weekend'] for df in dataframes])
vmin = np.floor(all_percent_changes.min())
vmax = np.ceil(all_percent_changes.max())

# Create the subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw={'projection': ccrs.PlateCarree()}, dpi=300)
axes = axes.flatten()
axes[-1].set_visible(False)

plt.subplots_adjust(wspace=0.3, hspace=0.2)

# Predefined suffixes for percentage change columns
suffixes = ['06_01', '11_01', '16_01', '22_01']

# Collect all percentage changes for scaling the marker size in the legend
all_pct_changes = []

# Plot each dataframe in a subplot
for i, (ax, df, title) in enumerate(zip(axes, dataframes, titles)):
    ax.set_extent([-111.3, -110.50, 31.75, 32.55], crs=ccrs.PlateCarree())  # Adjust the extent as needed
    ax.coastlines()

    # Add additional map features, such as rivers, borders, or land color
    ax.add_feature(cfeature.RIVERS)
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.LAND, facecolor='white')

    # Plot the shapefiles
    for gdf, color in zip([gdf_pima], ['gray']):
        ax.add_geometries(gdf.geometry, crs=ccrs.PlateCarree(), edgecolor=color, facecolor='none', linewidth=1)

    # Define the marker style for the first subplot (fixed size and up triangles)
    if i == 0:
        scatter = ax.scatter(df['Longitude'], df['Latitude'], c=df['weekend'], cmap='seismic', vmin=vmin, vmax=vmax,
                             edgecolors='k', zorder=35, s=500, marker='^')
    
    # For the remaining subplots, use different marker styles based on the change direction and scaled sizes
    else:
        pct_df = pct_changes[i-1]  # Get the corresponding percentage change DataFrame

        # Use predefined suffixes to match column names
        suffix = suffixes[i-1]

        for _, row in pct_df.iterrows():
            longitude = row['Longitude']
            latitude = row['Latitude']
            weekday_value = df.loc[df['Location'] == row['Location'], 'weekend'].values[0]
            weekday_pct_change = row[f'weekend_pct_diff_{suffix}']

            # Append percentage changes for legend scaling
            all_pct_changes.append(np.abs(weekday_pct_change))

            # Marker type: up triangle for positive change, down triangle for negative
            marker = '^' if weekday_pct_change > 0 else 'v'

            # Plot with scaled marker size
            marker_size = np.abs(weekday_pct_change) * 500  # Adjust scaling factor
            scatter = ax.scatter(longitude, latitude, c=[weekday_value], cmap='seismic', vmin=vmin, vmax=vmax, edgecolors='k',
                                 s=marker_size, marker=marker, zorder=35)

    # Add a title
    ax.set_title(title, fontsize=18)

    # Add state borders
    ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='black')

    for ax in axes[:-1]:
        # Add gridlines
        gl = ax.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
        gl.xlabels_top = False
        gl.ylabels_right = False
        gl.xlabel_style = {'size': 18}
        gl.ylabel_style = {'size': 18}

        # Hide labels on the right and top sides of the subplot
        gl.xlabels_top = False
        gl.ylabels_right = False

    '''for i in range(2):
        for j in range(2):
            axx = axes[i * 2 + j]  # Indexing as a 1D array
            gl = axx.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
            gl.xlabels_top = False
            gl.ylabels_right = False
            gl.xlabel_style = {'size': 18}
            gl.ylabel_style = {'size': 18}

            if i != 1:  
                gl.xlabels_bottom = False
            if j != 0:  
                gl.ylabels_left = False'''

    # Set gridlines
    #gl = ax.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
    #gl.xlabels_top = False
    #gl.ylabels_right = False
    #gl.xlabel_style = {'size': 14}
    #gl.ylabel_style = {'size': 14}

# Add a single colorbar for all subplots
cbar = fig.colorbar(scatter, ax=axes, orientation='horizontal', pad=0.05, extend='both', fraction=0.05, aspect=30)
cbar.ax.tick_params(labelsize=18)
cbar.set_label('MDA8 O$_3$ weekend (ppb)', fontsize=18)

# Calculate data-based percentage ranges using quantiles
low_range = (np.percentile(all_pct_changes, 0), np.percentile(all_pct_changes, 33))  # 0-33%
mid_range = (np.percentile(all_pct_changes, 34), np.percentile(all_pct_changes, 66))  # 34-66%
high_range = (np.percentile(all_pct_changes, 67), np.percentile(all_pct_changes, 100))  # 67-100%

# Custom legend for marker size (absolute percentage ranges)
legend_elements = [
    mlines.Line2D([], [], marker='^', color='w', label=f'{low_range[0]:.0f}% - {low_range[1]:.0f}%', markersize=low_range[1] * 10,
                  markerfacecolor='gray', markeredgewidth=2),
    mlines.Line2D([], [], marker='^', color='w', label=f'{mid_range[0]:.0f}% - {mid_range[1]:.0f}%', markersize=mid_range[1] * 9,
                  markerfacecolor='gray', markeredgewidth=2),
    mlines.Line2D([], [], marker='^', color='w', label=f'{high_range[0]:.0f}% +', markersize=high_range[1] * 7,
                  markerfacecolor='gray', markeredgewidth=2),
]

# Add custom legend for marker size, adjust position to avoid overlap
#plt.legend(handles=legend_elements, title="Percent Change (Absolute)", title_fontsize=14, labelspacing=1.0, handletextpad=0.2, loc='lower right', bbox_to_anchor=(1.2, 0.8), ncol=1, fontsize=14)
fig.legend(handles=legend_elements, title="Percent Change (Absolute)", title_fontsize=18, labelspacing=1.0, handletextpad=0.2,
           loc='center', bbox_to_anchor=(0.75, 0.35), fontsize=18, ncol=1, frameon=False)

plt.show()


In [ ]:
# List of dataframes
titles = ['2001-2005', '2006-2010', '2011-2015', '2016-2019', '2020-2022']

# Assume percentage change DataFrames have been calculated as:
# total_pct_change_05_97, total_pct_change_11_05, total_pct_change_17_11
pct_changes = [total_pct_change_06_01, total_pct_change_11_01, total_pct_change_16_01, total_pct_change_22_01]
dataframes = [Total_01, Total_06, Total_11, Total_16, Total_22]

# Normalize the colormap to have the middle at 0
all_percent_changes = pd.concat([df['weekday'] for df in dataframes])
vmin = np.floor(all_percent_changes.min())
vmax = np.ceil(all_percent_changes.max())

# Create the subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw={'projection': ccrs.PlateCarree()}, dpi=300)
axes = axes.flatten()
axes[-1].set_visible(False)

plt.subplots_adjust(wspace=0.3, hspace=0.20)

# Predefined suffixes for percentage change columns
suffixes = ['06_01', '11_01', '16_01', '22_01']

# Collect all percentage changes for scaling the marker size in the legend
all_pct_changes = []

# Plot each dataframe in a subplot
for i, (ax, df, title) in enumerate(zip(axes, dataframes, titles)):
    ax.set_extent([-111.3, -110.50, 31.75, 32.55], crs=ccrs.PlateCarree())  # Adjust the extent as needed
    ax.coastlines()

    # Add additional map features, such as rivers, borders, or land color
    ax.add_feature(cfeature.RIVERS)
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.LAND, facecolor='white')

    # Plot the shapefiles
    for gdf, color in zip([gdf_pima], ['gray']):
        ax.add_geometries(gdf.geometry, crs=ccrs.PlateCarree(), edgecolor=color, facecolor='none', linewidth=1)

    # Define the marker style for the first subplot (fixed size and up triangles)
    if i == 0:
        scatter = ax.scatter(df['Longitude'], df['Latitude'], c=df['weekday'], cmap='seismic', vmin=vmin, vmax=vmax,
                             edgecolors='k', zorder=35, s=500, marker='^')
    
    # For the remaining subplots, use different marker styles based on the change direction and scaled sizes
    else:
        pct_df = pct_changes[i-1]  # Get the corresponding percentage change DataFrame

        # Use predefined suffixes to match column names
        suffix = suffixes[i-1]

        for _, row in pct_df.iterrows():
            longitude = row['Longitude']
            latitude = row['Latitude']
            weekday_value = df.loc[df['Location'] == row['Location'], 'weekday'].values[0]
            weekday_pct_change = row[f'weekday_pct_diff_{suffix}']

            # Append percentage changes for legend scaling
            all_pct_changes.append(np.abs(weekday_pct_change))

            # Marker type: up triangle for positive change, down triangle for negative
            marker = '^' if weekday_pct_change > 0 else 'v'

            # Plot with scaled marker size
            marker_size = np.abs(weekday_pct_change) * 300  # Adjust scaling factor
            scatter = ax.scatter(longitude, latitude, c=[weekday_value], cmap='seismic', vmin=vmin, vmax=vmax, edgecolors='k',
                                 s=marker_size, marker=marker, zorder=35)

    # Add a title
    ax.set_title(title, fontsize=18)

    # Add state borders
    ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='black')

    for ax in axes[:-1]:
        # Add gridlines
        gl = ax.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
        gl.xlabels_top = False
        gl.ylabels_right = False
        gl.xlabel_style = {'size': 18}
        gl.ylabel_style = {'size': 18}

        # Hide labels on the right and top sides of the subplot
        gl.xlabels_top = False
        gl.ylabels_right = False

    '''for i in range(2):
        for j in range(2):
            axx = axes[i * 2 + j]  # Indexing as a 1D array
            gl = axx.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
            gl.xlabels_top = False
            gl.ylabels_right = False
            gl.xlabel_style = {'size': 18}
            gl.ylabel_style = {'size': 18}

            if i != 1:  
                gl.xlabels_bottom = False
            if j != 0:  
                gl.ylabels_left = False'''

    # Set gridlines
    #gl = ax.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
    #gl.xlabels_top = False
    #gl.ylabels_right = False
    #gl.xlabel_style = {'size': 14}
    #gl.ylabel_style = {'size': 14}

# Add a single colorbar for all subplots
cbar = fig.colorbar(scatter, ax=axes, orientation='horizontal', pad=0.05, extend='both', fraction=0.05, aspect=30)
cbar.ax.tick_params(labelsize=18)
cbar.set_label('MDA8 O$_3$ weekday (ppb)', fontsize=18)

# Calculate data-based percentage ranges using quantiles
low_range = (np.percentile(all_pct_changes, 0), np.percentile(all_pct_changes, 33))  # 0-33%
mid_range = (np.percentile(all_pct_changes, 34), np.percentile(all_pct_changes, 66))  # 34-66%
high_range = (np.percentile(all_pct_changes, 67), np.percentile(all_pct_changes, 100))  # 67-100%

# Custom legend for marker size (absolute percentage ranges)
legend_elements = [
    mlines.Line2D([], [], marker='^', color='w', label=f'{low_range[0]:.0f}% - {low_range[1]:.0f}%', markersize=low_range[1] * 8,
                  markerfacecolor='gray', markeredgewidth=2),
    mlines.Line2D([], [], marker='^', color='w', label=f'{mid_range[0]:.0f}% - {mid_range[1]:.0f}%', markersize=mid_range[1] * 7,
                  markerfacecolor='gray', markeredgewidth=2),
    mlines.Line2D([], [], marker='^', color='w', label=f'{high_range[0]:.0f}% +', markersize=high_range[1] * 4,
                  markerfacecolor='gray', markeredgewidth=2),
]

# Add custom legend for marker size, adjust position to avoid overlap
#plt.legend(handles=legend_elements, title="Percent Change (Absolute)", title_fontsize=14, labelspacing=1.0, handletextpad=0.2, handleheight = 1.2, loc='lower center', bbox_to_anchor=(0.72, 1.82), ncol=1, fontsize=14)
fig.legend(handles=legend_elements, title="Percent Change (Absolute)", title_fontsize=18, labelspacing=1.0, handletextpad=0.2,
           loc='center', bbox_to_anchor=(0.75, 0.35), fontsize=18, ncol=1, frameon=False)
plt.show()


In [ ]:
import matplotlib.colors as mcolors

# Function to calculate ranges for legend
def calculate_ranges(pct_changes):
    all_pct_values = []
    # Flatten all 'weekend_pct_diff' values across DataFrames into a single list
    for pct_df in pct_changes:
        all_pct_values.extend(pct_df.filter(regex='difference_pct_diff').abs().values.flatten())

    # Remove extreme outliers above a certain threshold
    capped_values = [v for v in all_pct_values if v <= 110]  # Adjust the cap as needed

    # Calculate dynamic range values based on actual data
    low_threshold = np.percentile(capped_values, 40)
    mid_threshold = np.percentile(capped_values, 80)
    high_threshold = np.max(capped_values)

    low_range = (0, low_threshold)
    mid_range = (low_threshold, mid_threshold)
    high_range = (mid_threshold, high_threshold)

    return low_range, mid_range, high_range

# Function to plot the maps
def plot_maps(dataframes, titles, shapefiles, pct_changes, suffixes):
    # Calculate the overall min and max values across all dataframes
    all_percent_changes = pd.concat([df['Difference'] for df in dataframes])
    vmin = np.floor(all_percent_changes.min())
    vmax = np.ceil(all_percent_changes.max())
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    cmap = plt.get_cmap('coolwarm')


    # Create the subplots
    fig, axes = plt.subplots(5, 5, figsize=(25, 25), subplot_kw={'projection': ccrs.PlateCarree()}, dpi=300)
    axes = axes.flatten()

    plt.subplots_adjust(wspace=0.3, hspace=0.1)

    # Define row labels
    row_labels = ['DJF', 'MAM', 'J', 'JA', 'SON']

    # Calculate dynamic ranges based on percentage changes
    low_range, mid_range, high_range = calculate_ranges(pct_changes)

    # Plot each dataframe in a subplot
    for idx, ax in enumerate(axes):
        df = dataframes[idx]

        ax.set_extent([-111.3, -110.50, 31.75, 32.55], crs=ccrs.PlateCarree())  # Adjust the extent as needed
        ax.coastlines()

        # Add additional map features, such as rivers, borders, or land color
        ax.add_feature(cfeature.RIVERS)
        ax.add_feature(cfeature.BORDERS)
        ax.add_feature(cfeature.LAND, facecolor='white')

        # Plot the shapefiles
        for gdf, color in zip(shapefiles, ['gray']):
            ax.add_geometries(gdf.geometry, crs=ccrs.PlateCarree(), edgecolor=color, facecolor='none', linewidth=1)

        # For the first column (no pct_changes), just plot triangles based on the dataframe's weekend values
        if idx % 5 == 0:  # First column in each row
            ax.scatter(df['Longitude'], df['Latitude'], 
                       c=df['Difference'], cmap=cmap, norm=norm, edgecolors='k', 
                       marker='^', s=400, zorder=40)  # Use fixed size triangles for the first column

        # Plot the triangle markers based on the pct_changes data for the last three columns
        else:
            row_idx = idx // 5  # Determine the row index (0-4)
            pct_idx = (idx % 5) - 1  # Determine which pct_change (0-2)
            pct_df = pct_changes[row_idx * 4 + pct_idx]  # Access the correct pct_change DataFrame
            suffix = suffixes[pct_idx]  # Use suffix based on the column index within the row
            for i, row in pct_df.iterrows():
                diff_pct_change = row[f'difference_pct_diff{suffix}']  # Correctly reference the pct_diff column

                # Set the marker type based on the percentage change
                marker = '^' if diff_pct_change > 0 else 'v'
                # Set the size of the marker based on the absolute percentage change
                #size = abs(diff_pct_change) * 100  # Adjust scaling factor as necessary
                max_marker_size = 1000  # Set a maximum limit for marker size
                scaling_factor = 10 
                marke_size = np.clip(np.abs(diff_pct_change) * scaling_factor, 50, max_marker_size)

                # Plot the triangle marker for the percentage change, color based on the weekend value in the dataframe
                ax.scatter(row['Longitude'], row['Latitude'], 
                           c=[df.loc[df['Location'] == row['Location'], 'Difference'].values[0]], 
                           cmap=cmap, norm=norm, edgecolors='k', marker=marker, 
                           s=marke_size, zorder=40)

        # Add a title only for the first row
        if idx < 5:
            ax.set_title(titles[idx], fontsize=18)

        # Add state borders
        ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='black')

        # Add row labels for the first column in each row
        if idx % 5 == 0:
            row_idx = idx // 5
            ax.annotate(row_labels[row_idx], xy=(-0.5, 0.5), xycoords='axes fraction',
                        size=18, ha='right', va='center', rotation=90, weight='bold')

    for i in range(5):
        for j in range(5):
            axx = axes[i * 5 + j]  # Indexing as a 1D array
            gl = axx.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
            gl.xlabels_top = False
            gl.ylabels_right = False
            gl.xlabel_style = {'size': 18}
            gl.ylabel_style = {'size': 18}


            if i != 4:  
                gl.xlabels_bottom = False
            if j != 0:  
                gl.ylabels_left = False


    # Add a single colorbar for all subplots
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, orientation='horizontal', pad=0.05, extend='both', fraction=0.05, aspect=30)
    cbar.ax.tick_params(labelsize=18)
    cbar.set_label('MDA8 O$_3$ Weekend-Weekday Difference (ppb)', fontsize=18)

    # Add a custom legend for the marker sizes representing absolute percentage changes
    legend_elements = [
        mlines.Line2D([], [], marker='^', color='w', label=f'{low_range[0]:.0f}% - {low_range[1]:.0f}%', markersize=low_range[1] * 0.4,
                      markerfacecolor='gray', markeredgewidth=2),
        mlines.Line2D([], [], marker='^', color='w', label=f'{mid_range[0]:.0f}% - {mid_range[1]:.0f}%', markersize=mid_range[1] * 0.3,
                      markerfacecolor='gray', markeredgewidth=2),
        mlines.Line2D([], [], marker='^', color='w', label=f'{high_range[0]:.0f}% +', markersize=high_range[1] * 0.4,
                      markerfacecolor='gray', markeredgewidth=2),
    ]

    # Add custom legend for marker size
    plt.legend(handles=legend_elements, title="Percent Change (Absolute)", title_fontsize=18, labelspacing=1.5, handletextpad=0.2, loc='lower center', bbox_to_anchor=(0.05, 5.6), ncol=3, fontsize=18)

    plt.show()

# Example usage
dataframes = [
    winter_season01, winter_season06, winter_season11, winter_season20, winter_season22,
    spring_season01, spring_season06, spring_season11, spring_season20, spring_season22,
    summer_season_dry01, summer_season_dry06, summer_season_dry11, summer_season_dry20, summer_season_dry22, 
    summer_season01, summer_season06, summer_season11, summer_season20, summer_season22,
    fall_season01, fall_season06, fall_season11, fall_season20, fall_season22
]

# Now including the correct three pct_changes per row
pct_changes = [
    winter_pct_change_06_01, winter_pct_change_11_01, winter_pct_change_20_01, winter_pct_change_22_01,
    spring_pct_change_06_01, spring_pct_change_11_01, spring_pct_change_20_01, spring_pct_change_22_01,
    sum_dry_pct_change_06_01, sum_dry_pct_change_11_01, sum_dry_pct_change_20_01,sum_dry_pct_change_22_01,
    summer_pct_change_06_01, summer_pct_change_11_01, summer_pct_change_20_01, summer_pct_change_22_01,
    fall_pct_change_06_01, fall_pct_change_11_01, fall_pct_change_20_01, fall_pct_change_22_01
]

# Suffixes for the pct_diff columns
suffixes = ['_06_01', '_11_01', '_20_01', '_22_01']

titles = ['2001-2005', '2006-2010', '2011-2015', '2016-2019', '2020-2022']
shapefiles = [gdf_pima]

plot_maps(dataframes, titles, shapefiles, pct_changes, suffixes)


In [ ]:
# Function to calculate ranges for legend
from matplotlib.colors import Normalize
def calculate_ranges(pct_changes):
    all_pct_values = []
    # Flatten all 'weekend_pct_diff' values across DataFrames into a single list
    for pct_df in pct_changes:
        all_pct_values.extend(pct_df.filter(regex='weekend_pct_diff').abs().values.flatten())

    # Remove extreme outliers above a certain threshold
    capped_values = [v for v in all_pct_values if v <= 200]  # Adjust the cap as needed

    # Calculate dynamic range values based on actual data
    low_threshold = np.percentile(capped_values, 40)
    mid_threshold = np.percentile(capped_values, 80)
    high_threshold = np.max(capped_values)

    low_range = (0, low_threshold)
    mid_range = (low_threshold, mid_threshold)
    high_range = (mid_threshold, high_threshold)

    return low_range, mid_range, high_range

# Function to plot the maps
def plot_maps(dataframes, titles, shapefiles, pct_changes, suffixes):
    # Calculate the overall min and max values across all dataframes
    all_percent_changes = pd.concat([df['weekend'] for df in dataframes])
    vmin = np.floor(all_percent_changes.min())
    vmax = np.ceil(all_percent_changes.max())
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap('seismic')

    # Create the subplots
    fig, axes = plt.subplots(5, 4, figsize=(20, 25), subplot_kw={'projection': ccrs.PlateCarree()}, dpi=300)
    axes = axes.flatten()

    plt.subplots_adjust(wspace=0.3, hspace=0.1)

    # Define row labels
    row_labels = ['DJF', 'MAM', 'J', 'JA', 'SON']

    # Calculate dynamic ranges based on percentage changes
    low_range, mid_range, high_range = calculate_ranges(pct_changes)

    # Plot each dataframe in a subplot
    for idx, ax in enumerate(axes):
        df = dataframes[idx]

        ax.set_extent([-111.3, -110.50, 31.75, 32.55], crs=ccrs.PlateCarree())  # Adjust the extent as needed
        ax.coastlines()

        # Add additional map features, such as rivers, borders, or land color
        ax.add_feature(cfeature.RIVERS)
        ax.add_feature(cfeature.BORDERS)
        ax.add_feature(cfeature.LAND, facecolor='white')

        # Plot the shapefiles
        for gdf, color in zip(shapefiles, ['gray']):
            ax.add_geometries(gdf.geometry, crs=ccrs.PlateCarree(), edgecolor=color, facecolor='none', linewidth=1)

        # For the first column (no pct_changes), just plot triangles based on the dataframe's weekend values
        if idx % 4 == 0:  # First column in each row
            ax.scatter(df['Longitude'], df['Latitude'], 
                       c=df['weekend'], cmap=cmap, norm=norm, edgecolors='k', 
                       marker='^', s=400, zorder=40)  # Use fixed size triangles for the first column

        # Plot the triangle markers based on the pct_changes data for the last three columns
        else:
            row_idx = idx // 4  # Determine the row index (0-4)
            pct_idx = (idx % 4) - 1  # Determine which pct_change (0-2)
            pct_df = pct_changes[row_idx * 3 + pct_idx]  # Access the correct pct_change DataFrame
            suffix = suffixes[pct_idx]  # Use suffix based on the column index within the row
            for i, row in pct_df.iterrows():
                weekend_pct_change = row[f'weekend_pct_diff{suffix}']  # Correctly reference the pct_diff column

                # Set the marker type based on the percentage change
                marker = '^' if weekend_pct_change > 0 else 'v'
                # Set the size of the marker based on the absolute percentage change
                size = abs(weekend_pct_change) * 200  # Adjust scaling factor as necessary

                # Plot the triangle marker for the percentage change, color based on the weekend value in the dataframe
                ax.scatter(row['Longitude'], row['Latitude'], 
                           c=[df.loc[df['Location'] == row['Location'], 'weekend'].values[0]], 
                           cmap=cmap, norm=norm, edgecolors='k', marker=marker, 
                           s=size, zorder=40)

        # Add a title only for the first row
        if idx < 4:
            ax.set_title(titles[idx], fontsize=18)

        # Add state borders
        ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='black')
        
        # Add row labels for the first column in each row
        if idx % 4 == 0:
            row_idx = idx // 4
            ax.annotate(row_labels[row_idx], xy=(-0.3, 0.5), xycoords='axes fraction',
                        size=18, ha='right', va='center', rotation=90, weight='bold')

        # Set gridlines
    for i in range(5):
        for j in range(4):
            axx = axes[i * 4 + j]  # Indexing as a 1D array
            gl = axx.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
            gl.xlabels_top = False
            gl.ylabels_right = False
            gl.xlabel_style = {'size': 18}
            gl.ylabel_style = {'size': 18}

            if i != 4:  
                gl.xlabels_bottom = False
            if j != 0:  
                gl.ylabels_left = False



    # Add a single colorbar for all subplots
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, orientation='horizontal', pad=0.05, extend='both', fraction=0.05, aspect=30)
    cbar.ax.tick_params(labelsize=20)
    cbar.set_label('MDA8 O$_3$ Weekend (ppb)', fontsize=20)

    # Add a custom legend for the marker sizes representing absolute percentage changes
    legend_elements = [
        mlines.Line2D([], [], marker='^', color='w', label=f'{low_range[0]:.0f}% - {low_range[1]:.0f}%', markersize=low_range[1] * 8,
                      markerfacecolor='gray', markeredgewidth=2),
        mlines.Line2D([], [], marker='^', color='w', label=f'{mid_range[0]:.0f}% - {mid_range[1]:.0f}%', markersize=mid_range[1] * 6,
                      markerfacecolor='gray', markeredgewidth=2),
        mlines.Line2D([], [], marker='^', color='w', label=f'{high_range[0]:.0f}% +', markersize=high_range[1] * 3.2,
                      markerfacecolor='gray', markeredgewidth=2),
    ]

    # Add custom legend for marker size
    plt.legend(handles=legend_elements, title="Percent Change (Absolute)", title_fontsize=18, labelspacing=1.5, handletextpad=0.2, loc='lower center', bbox_to_anchor=(0.05, 5.6), ncol=3, fontsize=18)

    plt.show()

# Example usage
dataframes = [
    winter_season01, winter_season06, winter_season11, winter_season20,
    spring_season01, spring_season06, spring_season11, spring_season20,
    summer_season_dry01, summer_season_dry06, summer_season_dry11, summer_season_dry20, 
    summer_season01, summer_season06, summer_season11, summer_season20,
    fall_season01, fall_season06, fall_season11, fall_season20
]

# Now including the correct three pct_changes per row
pct_changes = [
    winter_pct_change_06_01, winter_pct_change_11_01, winter_pct_change_20_01,
    spring_pct_change_06_01, spring_pct_change_11_01, spring_pct_change_20_01,
    sum_dry_pct_change_06_01, sum_dry_pct_change_11_01, sum_dry_pct_change_20_01,
    summer_pct_change_06_01, summer_pct_change_11_01, summer_pct_change_20_01,
    fall_pct_change_06_01, fall_pct_change_11_01, fall_pct_change_20_01
]

# Suffixes for the pct_diff columns
suffixes = ['_06_01', '_11_01', '_20_01']

titles = ['2001-2005', '2006-2010', '2011-2015', '2016-2020']
shapefiles = [gdf_pima]

plot_maps(dataframes, titles, shapefiles, pct_changes, suffixes)


In [ ]:
def calculate_ranges(pct_changes):
    all_pct_values = []
    # Flatten all 'weekend_pct_diff' values across DataFrames into a single list
    for pct_df in pct_changes:
        all_pct_values.extend(pct_df.filter(regex='weekday_pct_diff').abs().values.flatten())

    # Remove extreme outliers above a certain threshold
    capped_values = [v for v in all_pct_values if v <= 200]  # Adjust the cap as needed

    # Calculate dynamic range values based on actual data
    low_threshold = np.percentile(capped_values, 40)
    mid_threshold = np.percentile(capped_values, 80)
    high_threshold = np.max(capped_values)

    low_range = (0, low_threshold)
    mid_range = (low_threshold, mid_threshold)
    high_range = (mid_threshold, high_threshold)

    return low_range, mid_range, high_range

# Function to plot the maps
def plot_maps(dataframes, titles, shapefiles, pct_changes, suffixes):
    # Calculate the overall min and max values across all dataframes
    all_percent_changes = pd.concat([df['weekday'] for df in dataframes])
    vmin = np.floor(all_percent_changes.min())
    vmax = np.ceil(all_percent_changes.max())
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap('seismic')

    # Create the subplots
    fig, axes = plt.subplots(5, 4, figsize=(20, 25), subplot_kw={'projection': ccrs.PlateCarree()}, dpi=300)
    axes = axes.flatten()

    plt.subplots_adjust(wspace=0.3, hspace=0.1)

    # Define row labels
    row_labels = ['DJF', 'MAM', 'J', 'JA', 'SON']

    # Calculate dynamic ranges based on percentage changes
    low_range, mid_range, high_range = calculate_ranges(pct_changes)

    # Plot each dataframe in a subplot
    for idx, ax in enumerate(axes):
        df = dataframes[idx]

        ax.set_extent([-111.3, -110.50, 31.75, 32.55], crs=ccrs.PlateCarree())  # Adjust the extent as needed
        ax.coastlines()

        # Add additional map features, such as rivers, borders, or land color
        ax.add_feature(cfeature.RIVERS)
        ax.add_feature(cfeature.BORDERS)
        ax.add_feature(cfeature.LAND, facecolor='white')

        # Plot the shapefiles
        for gdf, color in zip(shapefiles, ['gray']):
            ax.add_geometries(gdf.geometry, crs=ccrs.PlateCarree(), edgecolor=color, facecolor='none', linewidth=1)

        # For the first column (no pct_changes), just plot triangles based on the dataframe's weekend values
        if idx % 4 == 0:  # First column in each row
            ax.scatter(df['Longitude'], df['Latitude'], 
                       c=df['weekday'], cmap=cmap, norm=norm, edgecolors='k', 
                       marker='^', s=400, zorder=40)  # Use fixed size triangles for the first column

        # Plot the triangle markers based on the pct_changes data for the last three columns
        else:
            row_idx = idx // 4  # Determine the row index (0-4)
            pct_idx = (idx % 4) - 1  # Determine which pct_change (0-2)
            pct_df = pct_changes[row_idx * 3 + pct_idx]  # Access the correct pct_change DataFrame
            suffix = suffixes[pct_idx]  # Use suffix based on the column index within the row
            for i, row in pct_df.iterrows():
                weekday_pct_change = row[f'weekday_pct_diff{suffix}']  # Correctly reference the pct_diff column

                # Set the marker type based on the percentage change
                marker = '^' if weekday_pct_change > 0 else 'v'
                # Set the size of the marker based on the absolute percentage change
                size = abs(weekday_pct_change) * 150  # Adjust scaling factor as necessary

                # Plot the triangle marker for the percentage change, color based on the weekend value in the dataframe
                ax.scatter(row['Longitude'], row['Latitude'], 
                           c=[df.loc[df['Location'] == row['Location'], 'weekday'].values[0]], 
                           cmap=cmap, norm=norm, edgecolors='k', marker=marker, 
                           s=size, zorder=40)

        # Add a title only for the first row
        if idx < 4:
            ax.set_title(titles[idx], fontsize=18)

        # Add state borders
        ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='black')

        # Add row labels for the first column in each row
        if idx % 4 == 0:
            row_idx = idx // 4
            ax.annotate(row_labels[row_idx], xy=(-0.3, 0.5), xycoords='axes fraction',
                        size=18, ha='right', va='center', rotation=90, weight='bold')

        # Add row labels for the first column in each row
        if idx % 4 == 0:
            row_idx = idx // 4
            ax.annotate(row_labels[row_idx], xy=(-0.3, 0.5), xycoords='axes fraction',
                        size=18, ha='right', va='center', rotation=90, weight='bold')

    for i in range(5):
        for j in range(4):
            axx = axes[i * 4 + j]  # Indexing as a 1D array
            gl = axx.gridlines(draw_labels=True, linewidth=0, color='gray', alpha=0.5, linestyle='--')
            gl.xlabels_top = False
            gl.ylabels_right = False
            gl.xlabel_style = {'size': 18}
            gl.ylabel_style = {'size': 18}

            if i != 4:  
                gl.xlabels_bottom = False
            if j != 0:  
                gl.ylabels_left = False


    # Add a single colorbar for all subplots
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, orientation='horizontal', pad=0.05, extend='both', fraction=0.05, aspect=30)
    cbar.ax.tick_params(labelsize=20)
    cbar.set_label('MDA8 O$_3$ Weekday (ppb)', fontsize=20)

    # Add a custom legend for the marker sizes representing absolute percentage changes
    legend_elements = [
        mlines.Line2D([], [], marker='^', color='w', label=f'{low_range[0]:.0f}% - {low_range[1]:.0f}%', markersize=low_range[1] * 8,
                      markerfacecolor='gray', markeredgewidth=2),
        mlines.Line2D([], [], marker='^', color='w', label=f'{mid_range[0]:.0f}% - {mid_range[1]:.0f}%', markersize=mid_range[1] * 3.5,
                      markerfacecolor='gray', markeredgewidth=2),
        mlines.Line2D([], [], marker='^', color='w', label=f'{high_range[0]:.0f}% +', markersize=high_range[1] * 1.5,
                      markerfacecolor='gray', markeredgewidth=2),
    ]

    # Add custom legend for marker size
    plt.legend(handles=legend_elements, title="Percent Change (Absolute)", title_fontsize=18, labelspacing=1.5, handletextpad=0.2, loc='lower center', bbox_to_anchor=(0.05, 5.6), ncol=3, fontsize=18)

    plt.show()

# Example usage
dataframes = [
    winter_season01, winter_season06, winter_season11, winter_season20,
    spring_season01, spring_season06, spring_season11, spring_season20,
    summer_season_dry01, summer_season_dry06, summer_season_dry11, summer_season_dry20, 
    summer_season01, summer_season06, summer_season11, summer_season20,
    fall_season01, fall_season06, fall_season11, fall_season20
]

# Now including the correct three pct_changes per row
pct_changes = [
    winter_pct_change_06_01, winter_pct_change_11_01, winter_pct_change_20_01,
    spring_pct_change_06_01, spring_pct_change_11_01, spring_pct_change_20_01,
    sum_dry_pct_change_06_01, sum_dry_pct_change_11_01, sum_dry_pct_change_20_01,
    summer_pct_change_06_01, summer_pct_change_11_01, summer_pct_change_20_01,
    fall_pct_change_06_01, fall_pct_change_11_01, fall_pct_change_20_01
]

# Suffixes for the pct_diff columns
suffixes = ['_06_01', '_11_01', '_20_01']

titles = ['2001-2005', '2006-2010', '2011-2015', '2016-2020']
shapefiles = [gdf_pima]

plot_maps(dataframes, titles, shapefiles, pct_changes, suffixes)
